In [ ]:
# ==============================================================================
# KROK 1: Import bibliotek i wygenerowanie przykładowego zbioru biznesowego
# ==============================================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Ustawienia estetyczne wykresów
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 5)

# Tworzymy syntetyczny zbiór danych: Ceny mieszkań z błędami systemowymi
np.random.seed(42)
n_samples = 250

data = {
    'Metraz': np.random.normal(60, 20, n_samples).round(1),
    'Pokoje': np.random.choice([1, 2, 3, 4], size=n_samples, p=[0.2, 0.4, 0.3, 0.1]),
    'Lokalizacja': np.random.choice(['Centrum', 'Mokotow', 'Wola', 'Praga'], size=n_samples),
    'Rok_Budowy': np.random.randint(1970, 2024, size=n_samples),
    'Standard': np.random.choice(['Niski', 'Sredni', 'Wysoki'], size=n_samples),
    'Cena_PLN': np.zeros(n_samples)
}

df = pd.DataFrame(data)

# Realistyczna formuła cenowa + szum
df['Cena_PLN'] = (df['Metraz'] * 12000 + df['Pokoje'] * 25000 + np.random.normal(0, 50000, n_samples)).round(-3)

# --- CELOWE WPROWADZENIE "BRUDU" (Typowe błędy w bazach firmowych) ---
# 1. Braki danych (NaN)
df.loc[np.random.choice(df.index, 15, replace=False), 'Metraz'] = np.nan
df.loc[np.random.choice(df.index, 10, replace=False), 'Rok_Budowy'] = np.nan

# 2. Outliery (Błędy ludzkie/systemowe)
df.loc[5, 'Metraz'] = 999.0          # Ktoś wpisał kod błędu jako metraż
df.loc[12, 'Cena_PLN'] = 15_000_000   # Błąd o jedno zero za dużo (15 mln za 50m2)
df.loc[45, 'Cena_PLN'] = -50_000      # Ujemna cena (np. błędna korekta faktury)

print("✅ Zbiór danych został załadowany pomyślnie. Rozmiar:", df.shape)
df.head()

In [ ]:
# ==============================================================================
# KROK 2: Wstępna inspekcja techniczna
# ==============================================================================
print("--- 1. Typy danych i braki niepuste (info) ---")
df.info()

print("\n--- 2. Podsumowanie braków danych (NaN) ---")
missing_values = df.isnull().sum()
print(missing_values[missing_values > 0])

print("\n--- 3. Statystyki opisowe zmiennych numerycznych ---")
df.describe().T

In [ ]:
# ==============================================================================
# KROK 3: Wizualna detekcja wartości odstających (Boxplot i Scatter)
# ==============================================================================
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Wykres pudełkowy dla Ceny
sns.boxplot(x=df['Cena_PLN'], ax=axes[0], color='salmon')
axes[0].set_title("Wykrywanie Outlierów w Cenie (Boxplot)")

# Zależność Metraż vs Cena z widocznymi błędami
sns.scatterplot(data=df, x='Metraz', y='Cena_PLN', hue='Lokalizacja', ax=axes[1], s=70)
axes[1].set_title("Metraż vs Cena (Zwróć uwagę na punkty w skrajnych rogach)")

plt.tight_layout()
plt.show()

In [ ]:
# ==============================================================================
# KROK 4: Zastosowanie filtrów biznesowych
# ==============================================================================
# Usuwamy ewidentne błędy systemowe
df_cleaned = df[
    (df['Cena_PLN'] > 100_000) & (df['Cena_PLN'] < 5_000_000) &  # Rozsądny przedział cenowy
    ((df['Metraz'] < 250) | (df['Metraz'].isna()))                # Zachowujemy NaN na metrażu (obsłużymy je w pipeline)
].copy()

print(f"Liczba usuniętych rekordów skrajnie błędnych: {len(df) - len(df_cleaned)}")

# Sprawdzenie korelacji po wstępnym oczyszczeniu
plt.figure(figsize=(6, 4))
sns.heatmap(df_cleaned.select_dtypes(include=np.number).corr(), annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Macierz korelacji po usunięciu anomalii")
plt.show()

In [ ]:
# ==============================================================================
# KROK 5: Pipeline przygotowania danych: Imputacja + One-Hot Encoding
# ==============================================================================
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Podział na cechy (X) i target (y)
X = df_cleaned.drop(columns=['Cena_PLN'])
y = df_cleaned['Cena_PLN']

# Definiujemy, które kolumny jak traktujemy
num_features = ['Metraz', 'Pokoje', 'Rok_Budowy']
cat_features = ['Lokalizacja', 'Standard']

# Budujemy transformator dla kolumn numerycznych (Imputacja medianą + Skalowanie)
num_pipeline = ColumnTransformer([
    ('imputer', SimpleImputer(strategy='median'), num_features)
])

# Budujemy pełen proces przygotowania danych (ColumnTransformer)
preprocessor = ColumnTransformer(
    transformers=[
        ('num', SimpleImputer(strategy='median'), num_features),
        ('cat', OneHotEncoder(drop='first', sparse_output=False), cat_features)
    ]
)

# Przetwarzamy surowe dane X na macierz gotową do uczenia maszynowego
X_processed = preprocessor.fit_transform(X)

# Pobieramy nowe nazwy kolumn po One-Hot Encodingu
feature_names = num_features + list(preprocessor.named_transformers_['cat'].get_feature_names_out(cat_features))
X_final_df = pd.DataFrame(X_processed, columns=feature_names)

print("🚀 Dane gotowe do wejścia w model ML! (Brak NaN, same liczby, cechy zakodowane):")
X_final_df.head()